# 02 - Multi-Table Relational EDA & Aggregations

**Track 2 Lead**: Open / In Progress  
**Course**: UC Berkeley DATASCI 207 (Machine Learning at Scale)  
**Target Deliverable**: Project Milestone & Final Submission  
**Scope**: Multi-table relational joins across `bureau.csv`, `previous_application.csv`, `POS_CASH_balance.csv`, `installments_payments.csv`, and `credit_card_balance.csv`.

---

### Environment & Data Ingestion (Google Colab)
Run the cell below if executing in Google Colab to mount Google Drive and pull dataset files to `/content/local_data`.


In [ ]:
# NOTE FOR TEAMMATES:
# If you get 'Directory not found', add the shared folder to your My Drive:
# 1. Go to Google Drive (https://drive.google.com).
# 2. Click 'Shared with me' on the left menu.
# 3. Right-click the 'DATASCI207_finalproject' folder.
# 4. Select Organize -> Add shortcut (or press Shift + Z).
# 5. Choose 'My Drive' and click Add.

try:
    from google.colab import drive
    import shutil, os

    # Mount Google Drive
    drive.mount('/content/drive')

    # Copy shared project folder to local Colab storage
    src = "/content/drive/MyDrive/DATASCI207_finalproject"
    dst = "/content/DATASCI207_finalproject"

    shutil.copytree(src, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns('*.gdoc', '*.gsheet', '*.gslides'))
    print("Downloaded files:", os.listdir(dst))
except ImportError:
    print("Running in local environment; skipping Google Drive mount.")


### Cleaned Static Baseline Ingestion (Task 1)

Loads the cleaned and engineered static applicant dataset (`application_train_cleaned.parquet`) produced by Task 1.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# 1. Load Task 1 cleaned application matrix
data_path = Path('/content/DATASCI207_finalproject/data/processed/application_train_cleaned.parquet')
if not data_path.exists():
    data_path = Path('data/processed/application_train_cleaned.parquet')

df_cleaned = pd.read_parquet(data_path)
print(f'Loaded cleaned data: {df_cleaned.shape[0]:,} rows x {df_cleaned.shape[1]:,} columns')

# 2. Raw relational tables directory (bureau, previous_application, etc.)
raw_dir = Path('/content/DATASCI207_finalproject/data/raw/home_credit_default_risk')
if not raw_dir.exists():
    raw_dir = Path('data/raw/home_credit_default_risk')
RAW_DIR = raw_dir
if RAW_DIR.exists():
    print('Available raw tables:', [f.name for f in RAW_DIR.glob('*.csv')])

In [3]:
import gc

def downcast(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype('category')
    return df

def mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / 1e6



In [5]:
## aggregation 1 - bureau.csv - external credit bureau lines
bureau = pd.read_csv(
    RAW_DIR / 'bureau.csv',
    usecols=['SK_ID_CURR', 'CREDIT_ACTIVE', 'AMT_CREDIT_SUM_OVERDUE',
             'CREDIT_DAY_OVERDUE', 'AMT_CREDIT_SUM_DEBT', 'DAYS_CREDIT']
)
print(f"bereau.csv loaded: {bureau.shape[0]:,} rows, {mb(bureau):.1f} MB")
bureau = downcast(bureau)
print(f"after downcast: {mb(bureau):.1f} MB")

bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    BUREAU_ACTIVE_LINES=('CREDIT_ACTIVE', lambda s: (s == 'Active').sum()),
    BUREAU_TOTAL_LINES=('CREDIT_ACTIVE', 'count'),
    BUREAU_TOTAL_OVERDUE_DEBT=('AMT_CREDIT_SUM_OVERDUE', 'sum'),
    BUREAU_MAX_DAYS_OVERDUE=('CREDIT_DAY_OVERDUE', 'max'),
    BUREAU_TOTAL_DEBT=('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUREAU_OLDEST_CREDIT_DAYS=('DAYS_CREDIT', 'min'),
).reset_index()

del bureau
gc.collect()
bureau_agg.head()

bereau.csv loaded: 1,716,428 rows, 92.7 MB
after downcast: 42.9 MB


/var/folders/y4/6v54dls90g9_fpz5vx1pzmnh0000gn/T/ipykernel_6977/35998443.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


,SK_ID_CURR,BUREAU_ACTIVE_LINES,BUREAU_TOTAL_LINES,BUREAU_TOTAL_OVERDUE_DEBT,BUREAU_MAX_DAYS_OVERDUE,BUREAU_TOTAL_DEBT,BUREAU_OLDEST_CREDIT_DAYS
0,100001,3,7,0.0,0,596686.5,-1572
1,100002,2,8,0.0,0,245781.0,-1437
2,100003,1,4,0.0,0,0.0,-2586
3,100004,0,2,0.0,0,0.0,-1326
4,100005,2,3,0.0,0,568408.5,-373


In [7]:
## aggregation 2 - previous_application.csv - prior home credit applications
prev = pd.read_csv(
    RAW_DIR / 'previous_application.csv',
    usecols=['SK_ID_CURR', 'NAME_CONTRACT_STATUS', 'AMT_DOWN_PAYMENT', 'AMT_CREDIT']
)
print(f"previous_application.csv loaded: {prev.shape[0]:,} rows, {mb(prev):.1f} MB")
prev = downcast(prev)

prev['DOWN_PAYMENT_PCT'] = prev['AMT_DOWN_PAYMENT'] / prev['AMT_CREDIT']
prev['DOWN_PAYMENT_PCT'] = prev['DOWN_PAYMENT_PCT'].replace([np.inf, -np.inf], np.nan)

prev_agg = prev.groupby('SK_ID_CURR').agg(
    PREV_REFUSAL_COUNT=('NAME_CONTRACT_STATUS', lambda s: (s == 'Refused').sum()),
    PREV_APPLICATION_COUNT=('NAME_CONTRACT_STATUS', 'count'),
    PREV_APPROVAL_RATE=('NAME_CONTRACT_STATUS', lambda s: (s == 'Approved').mean()),
    PREV_MEAN_DOWN_PAYMENT_PCT=('DOWN_PAYMENT_PCT', 'mean'),
).reset_index()

del prev
gc.collect()
prev_agg.head()

previous_application.csv loaded: 1,670,214 rows, 66.6 MB


/var/folders/y4/6v54dls90g9_fpz5vx1pzmnh0000gn/T/ipykernel_6977/35998443.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


,SK_ID_CURR,PREV_REFUSAL_COUNT,PREV_APPLICATION_COUNT,PREV_APPROVAL_RATE,PREV_MEAN_DOWN_PAYMENT_PCT
0,100001,0,1,1.0,0.105940
1,100002,0,1,1.0,0.000000
2,100003,0,3,1.0,0.050585
3,100004,0,1,1.0,0.241719
4,100005,0,2,0.5,0.111173
